# `qwen3-8b-4bit-qlora-s00-r1` — DGX Spark recipe (PREPARED, not executed)

Controlled size-ablation successor to the executed `qwen3-4b-4bit-qlora-s00-r0`
baseline (69.2% on the 464-row scorable test split). Same corpus anchor, seed,
hyperparameters, target modules, LoRA rank/alpha, and trainer config. Only the
base model changes: `unsloth/Qwen3-4B-Instruct-2507` → `unsloth/Qwen3-8B`.

Container-aware. Run inside `nvcr.io/nvidia/pytorch:25.11-py3` (launched by
`models/recipes/dgx_spark/launch.sh`). Substrate-of-record: NVIDIA DGX Spark
Unsloth playbook. Hardware notes: `.meta/hardware.md`.

**Why Qwen3-8B as r1** (Leo, 2026-05-21):
- **Same family/tokenizer as r0** — Qwen3 lineage means the comparison isolates
  the size variable. A jump to a different family (Llama-3.1-8B, Mistral-7B,
  DeepSeek-distill) would confound size with tokenizer + pretraining mix.
- **Fits comfortably on DGX Spark UMA (119.6 GB)** — 8B 4-bit base ~5 GB +
  activations + optimizer state + LoRA grads = comfortable. The dormant Windows
  row's 16 GB VRAM is tight for 8B even at 4-bit, so this recipe is intentionally
  DGX-Spark-targeted (§3 item #5: "Deferred — needs DGX Spark ACTIVE").
- **Thinking capability preserved, not trained** — Qwen3-8B is hybrid
  (instruct + thinking via chat template). The Spiceland gold has no `<think>`
  traces; training with thinking ON would teach the model to emit empty think
  blocks and damage the prior. We render the chat template with
  `enable_thinking=False`, leaving the thinking weights pristine for downstream
  inference-time use. The size ablation is clean against r0 (which was
  Instruct-2507, no thinking mode).

**Decision log** (Leo, 2026-05-21):
- **Supersession-note rendering**: STILL DEFERRED (was r1 target; pushed to r2+).
  Diana's split JSONL drops `meta.gaap_supersession` and `eval/_format.py` does
  not template-render it into the assistant turn. r1 trains on raw Spiceland
  gold, same as r0. Resolving this requires re-emitting splits with a new sha.
- **Loss weighting 0.85/0.15**: STILL DEFERRED. SFTTrainer does not support
  per-segment loss weighting; a custom collator emitting per-token weights is
  required. Gated on the supersession block being plumbed into split records.
- **Reasoning-prior ablation (r2 target)**: replace Qwen3-8B with
  `unsloth/DeepSeek-R1-0528-Qwen3-8B`. Same tokenizer, same parameter count,
  swap-in reasoning prior. Isolates "thinking prior" against this r1's "plain
  Qwen3-8B" baseline. Separate run id.
- **r1 launch trigger**: user-gated. Recipe is *prepared*, not *executed*.
  Requires the dormant DGX Spark row to be ACTIVE again.

## §1 — Substrate guardrails

Fail fast if we are on the wrong host, wrong CUDA arch, wrong split version, or the seed split sha drifted from `eval/sft/splits/manifest.json`. Then apply project-wide deterministic seeding via `eval/seeding.py` (see the seeding-policy memory: `seedhash` is the sole seed-derivation mechanism, covering CPU + single-GPU + multi-GPU one-host + multi-host). Same guardrails as r0 — only the `RUN_ID` differs; the seeding call is new (r0 relied implicitly on `SFTConfig(seed=...)`).

In [ ]:
import hashlib, json, os, sys
from pathlib import Path

# Repo-root discovery: prefer the container mount `/workspace` if it exists
# (this notebook is designed to run inside nvcr.io/nvidia/pytorch:25.11-py3),
# otherwise walk up from cwd looking for a .git/ marker so host-side smoke
# tests work on Windows/macOS/Linux without a code change.
def _find_repo() -> Path:
    workspace = Path("/workspace")
    if workspace.is_dir() and (workspace / "eval" / "sft" / "splits").is_dir():
        return workspace
    forced = os.environ.get("REPO_ROOT")
    if forced:
        return Path(forced).resolve()
    p = Path.cwd().resolve()
    for d in [p, *p.parents]:
        if (d / ".git").is_dir() or (d / "eval" / "sft" / "splits").is_dir():
            return d
    raise RuntimeError(f"could not find repo root from {p}")

REPO = _find_repo()
print(f"REPO: {REPO}")
SEED_DIR = REPO / "eval/sft/splits/seed_00__351199285"
MANIFEST_PATH = REPO / "eval/sft/splits/manifest.json"
RUN_ID = "qwen3-8b-4bit-qlora-s00-r1"
RUN_DIR = REPO / f"models/runs/dgx_spark/{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

import torch
assert torch.cuda.is_available(), "CUDA not visible — launch via launch.sh with --gpus all"
cc = torch.cuda.get_device_capability(0)
name = torch.cuda.get_device_name(0)
print(f"device: {name}  compute_cap={cc[0]}.{cc[1]}")
assert cc == (12, 1), f"expected GB10 sm_120 (12,1), got {cc}"
assert "GB10" in name, f"expected GB10 device, got {name!r}"

for split in ("train", "valid", "test"):
    p = SEED_DIR / f"{split}.jsonl"
    assert p.exists(), f"missing {p}"

manifest = json.loads(MANIFEST_PATH.read_text())
assert manifest["source_jsonl_sha256"].startswith("fed6eb17de8be1e4"), (
    f"corpus drift: manifest source sha = {manifest['source_jsonl_sha256'][:16]}, "
    "expected spiceland9e-v1.1.0 = fed6eb17de8be1e4\u2026"
)
seed00 = next(s for s in manifest["seeds"] if s["seed_index"] == 0)
for split, want in seed00["file_sha256"].items():
    got = hashlib.sha256((SEED_DIR / f"{split}.jsonl").read_bytes()).hexdigest()
    assert got == want, f"{split} sha drift: got {got[:16]}\u2026 want {want[:16]}\u2026"
print("seed_00 splits sha-verified against manifest:")
for split, want in seed00["file_sha256"].items():
    print(f"  {split:<5} {want[:16]}\u2026")
print("corpus anchor: spiceland9e-v1.1.0")
print(f"counts: {seed00['split_counts']}")

# ── Deterministic seeding (project policy: seedhash is the sole derivation
# mechanism; eval/seeding.py covers CPU + single-GPU + multi-GPU one-host +
# multi-host topologies). Embed the report in manifest.json for replay.
sys.path.insert(0, str(REPO / "eval"))
from seeding import seed_everything  # noqa: E402
SEEDING_REPORT = seed_everything(seed_int=351199285)
print(f"seeding: topology={SEEDING_REPORT.topology} "
      f"rank={SEEDING_REPORT.rank}/{SEEDING_REPORT.world_size} "
      f"per_rank_seed={SEEDING_REPORT.per_rank_seed_int} "
      f"strict={SEEDING_REPORT.strict}")

## §2 — Dataset + tokenizer

Diana's splits already carry rendered `messages` (system + user + assistant) per `eval/_format.py`. We map those to a single `text` field via the Qwen3 chat template.

**r1 delta from r0**: Qwen3-8B (original, not the 2507 Instruct line) is a hybrid thinking model. We pass `enable_thinking=False` to `apply_chat_template` so the rendered text has no `<think></think>` block. The gold answers carry no thinking traces; training with thinking ON would damage the thinking prior. Inference-time thinking remains togglable after fine-tuning.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

BASE_MODEL_PRIMARY = "unsloth/Qwen3-8B"
BASE_MODEL_FALLBACK = "Qwen/Qwen3-8B"
MAX_SEQ_LEN = 2048

tok_probe = AutoTokenizer.from_pretrained(BASE_MODEL_PRIMARY, trust_remote_code=False)
TOKENIZER_REVISION = getattr(tok_probe, "name_or_path", BASE_MODEL_PRIMARY)
print(f"tokenizer probe ok: {TOKENIZER_REVISION}")
print(f"chat template present: {bool(tok_probe.chat_template)}")

ds = load_dataset(
    "json",
    data_files={
        "train": str(SEED_DIR / "train.jsonl"),
        "valid": str(SEED_DIR / "valid.jsonl"),
        "test":  str(SEED_DIR / "test.jsonl"),
    },
)
print({k: len(v) for k, v in ds.items()})

def render_text(rec):
    # rec["messages"] = [{role, content}, ...]; apply Qwen3 chat template with
    # the assistant turn fully present so train_on_responses_only can mask the
    # prompt half. Do not add a generation prompt.
    # enable_thinking=False: Qwen3-8B is hybrid; the Spiceland gold has
    # no <think> traces, so we render in non-thinking mode to preserve
    # the thinking prior. See decision log in the header cell.
    return tok_probe.apply_chat_template(
        rec["messages"], tokenize=False, add_generation_prompt=False,
        enable_thinking=False,
    )

ds = ds.map(lambda r: {"text": render_text(r)}, num_proc=4)
print("sample (first 600 chars of train[0].text):")
print(ds["train"][0]["text"][:600])

## §3 — Base model (4-bit QLoRA via Unsloth `FastModel`)

Playbook-validated path: `FastModel.from_pretrained(load_in_4bit=True, full_finetuning=False)`. Try the Unsloth pre-quantized repo first; on miss, fall back to the official Qwen repo and let Unsloth auto-quantize. Same call shape as r0; only the model name changes.

In [ ]:
from unsloth import FastModel, FastLanguageModel

BASE_MODEL_USED = BASE_MODEL_PRIMARY
try:
    model, tokenizer = FastModel.from_pretrained(
        model_name=BASE_MODEL_PRIMARY,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
        load_in_8bit=False,
        full_finetuning=False,
    )
except Exception as e:
    print(f"primary load failed ({type(e).__name__}: {e!s:.200}); falling back to {BASE_MODEL_FALLBACK}")
    BASE_MODEL_USED = BASE_MODEL_FALLBACK
    model, tokenizer = FastModel.from_pretrained(
        model_name=BASE_MODEL_FALLBACK,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
        load_in_8bit=False,
        full_finetuning=False,
    )
TOKENIZER_REVISION = getattr(tokenizer, "name_or_path", BASE_MODEL_USED)
print(f"base loaded: {BASE_MODEL_USED}")
print(f"tokenizer:   {TOKENIZER_REVISION}")

## §4 — LoRA adapter

Same rank/alpha/target-modules as r0 — the ablation against r0 must keep the adapter shape constant so the only delta is base-model size. Trainable-param count will scale with the 8B base hidden size.

In [ ]:
SEED = 351199285
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=TARGET_MODULES,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

## §5 — SFTTrainer config

Identical to r0 (per-device batch 2 × grad-accum 4 → effective 8; 3 epochs; cosine LR with 5% warmup; `adamw_8bit`; `eval_steps=110` for ≈12 dev-loss readings + EarlyStopping patience=3). The size ablation requires every knob below the base model to stay constant.

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

OUTPUT_DIR = str(RUN_DIR)

sft_cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.05,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    weight_decay=0.0,
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    dataset_text_field="text",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=110,
    save_strategy="steps",
    save_steps=110,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    fp16=False,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["valid"],
    args=sft_cfg,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

## §6 — `train_on_responses_only` masking

Mask the prompt half so the loss is computed only on the assistant turn. Qwen3 instruction templates are `<|im_start|>user` ... `<|im_start|>assistant` ... `<|im_end|>`.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
print("responses-only masking applied (Qwen3 im_start/im_end tags)")

## §7 — Train

**Wall-time projection** (r0 actuals → r1 estimate): r0 trained at 1.933 samples/s on this host (full real-world rate, including eval + checkpoint I/O), taking 91.5 min for 3,537 train rows × 3 epochs. Qwen3-8B has ~2× the active parameters per forward pass at the same batch size; expect ~0.95–1.05 samples/s and **~180–200 min wall-clock** (3.0–3.3 h). The UMA pool absorbs the extra activation memory; no batch reduction needed.

If throughput drops below 0.8 samples/s, run the playbook page-cache flush *between* training launches (UMA OOM-recovery):

```
sudo sh -c 'sync; echo 3 > /proc/sys/vm/drop_caches'
```

In [ ]:
import time
t0 = time.time()
train_result = trainer.train()
wall_clock_s = time.time() - t0
print(f"train done in {wall_clock_s/60:.1f} min")
print(train_result.metrics)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"adapter + tokenizer saved \u2192 {OUTPUT_DIR}")

## §8 — Manifest emission

Per Leo's standing rule — every run pinned. The `source_jsonl_sha256` binds this adapter to `spiceland9e-v1.1.0`, identical to r0; the cross-run delta is recoverable from the `recipe.base_model` field.

In [ ]:
import datetime as dt

def file_sha256(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

adapter_path = RUN_DIR / "adapter_model.safetensors"
if not adapter_path.exists():
    candidates = list(RUN_DIR.glob("adapter_model*"))
    adapter_path = candidates[0] if candidates else None
adapter_sha = file_sha256(adapter_path) if adapter_path and adapter_path.exists() else None

best_dev_loss = None
log_history = getattr(trainer.state, "log_history", [])
for entry in log_history:
    if "eval_loss" in entry:
        v = entry["eval_loss"]
        if best_dev_loss is None or v < best_dev_loss:
            best_dev_loss = v

train_samples = len(ds["train"])
throughput = (train_result.metrics.get("train_samples_per_second")
              if train_result and train_result.metrics else None)

manifest_out = {
    "run_id": RUN_ID,
    "created_at": dt.datetime.utcnow().isoformat() + "Z",
    "base_model": BASE_MODEL_USED,
    "tokenizer_revision": TOKENIZER_REVISION,
    "adapter_sha256": adapter_sha,
    "source_jsonl_sha256": manifest["source_jsonl_sha256"],
    "corpus_anchor": "spiceland9e-v1.1.0",
    "train_split_sha256": seed00["file_sha256"]["train"],
    "valid_split_sha256": seed00["file_sha256"]["valid"],
    "holdout_split_sha256": seed00["file_sha256"]["test"],
    "seed_int": SEED,
        "seeding_report": SEEDING_REPORT.to_dict(),
    "recipe": {
        "method": "qlora-4bit",
        "rank": 16,
        "alpha": 32,
        "dropout": 0,
        "target_modules": TARGET_MODULES,
        "max_seq_length": MAX_SEQ_LEN,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-4,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.05,
        "num_train_epochs": 3,
        "optim": "adamw_8bit",
        "bf16": True,
        "loss_weighting": "uniform",
        "loss_weighting_note": (
            "r1 inherits r0's uniform loss weighting. Pat's 0.85/0.15 gold/supersession "
            "weighting still deferred (no SFTTrainer support; needs custom collator + "
            "supersession block plumbed into split records)."
        ),
        "supersession_render": "absent",
        "supersession_note": (
            "r1 trains on raw Spiceland gold, same as r0. eval/_format.py does not render "
            "meta.gaap_supersession into the assistant turn, and Diana's split JSONL drops "
            "the field. Deferred to r2+ (needs split re-emission with a new sha)."
        ),
        "responses_only_masking": True,
    },
    "final_dev_loss": best_dev_loss,
    "final_train_loss": train_result.metrics.get("train_loss") if train_result else None,
    "wall_clock_seconds": wall_clock_s,
    "throughput_samples_per_second": throughput,
    "train_samples": train_samples,
    "gpu_device": name,
    "cuda_compute_cap": f"{cc[0]}.{cc[1]}",
    "container_image": "nvcr.io/nvidia/pytorch:25.11-py3",
    "unsloth_version": "2026.5.5",
    "trl_version": "0.26.1",
    "datasets_version": "4.3.0",
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest_out, indent=2) + "\n")
print(json.dumps(manifest_out, indent=2))

## §9 — Quick eval (50-sample probe)

NOT production eval. Vera's `eval/run.py` is the production path. This is a sanity-check that the adapter generates sensible completions before the user gates a full Vera run. Identical probe protocol to r0 so the side-by-side comparison is apples-to-apples.

**r1 delta**: same `enable_thinking=False` on the prompt template as §2 — training-time and inference-time chat-template rendering must match or the model will see a token distribution it was never trained on.

In [ ]:
import random
FastLanguageModel.for_inference(model)

rng = random.Random(SEED)
test_rows = list(ds["test"])
probe = rng.sample(test_rows, k=min(50, len(test_rows)))
probe_path = RUN_DIR / "probe_predictions.jsonl"

with probe_path.open("w") as fh:
    for i, rec in enumerate(probe):
        msgs = rec["messages"][:2]
        prompt_text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        out = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
        completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        row = {
            "id": rec["meta"]["id"],
            "chapter": rec["meta"]["chapter"],
            "type": rec["meta"]["type"],
            "primary_bloom": rec["meta"]["primary_bloom"],
            "gold_assistant": rec["messages"][2]["content"],
            "prediction": completion,
        }
        fh.write(json.dumps(row) + "\n")
        if i < 3:
            print(f"--- probe {i} [{row['id']}, {row['type']}] ---")
            print(f"GOLD : {row['gold_assistant'][:200]}")
            print(f"PRED : {row['prediction'][:200]}")
print(f"probe predictions \u2192 {probe_path}")

## §10 — GGUF export (optional, gated)

For Ollama / llama.cpp portability. Default OFF for r1 (this is a prepared-not-executed stub); flip to True after a Vera PASS on the adapter.

In [ ]:
do_gguf_export = False

if do_gguf_export:
    gguf_dir = RUN_DIR / "gguf"
    gguf_dir.mkdir(exist_ok=True)
    model.save_pretrained_gguf(
        str(gguf_dir),
        tokenizer,
        quantization_method="q4_k_m",
    )
    print(f"GGUF q4_k_m \u2192 {gguf_dir}")
else:
    print("GGUF export skipped (flag off). Enable by setting do_gguf_export=True.")